## Sentiment Analysis
#### Sentiment Schema design
I think that a good way to approach the design of a sentiment schema for this exercise is to identify the important categories that could be discussed in a review. In my opinion, the important topics are:
- **product_quality**: This is the most important sentiment category, as it is what other customers will be looking for the most from reviews.
- **customer_service**: This is the category that many customers will evaluate to determine if they want to make a purchase.
- **product_ease_of_use**: This category refers to all issues or good experiences related to products other than the quality (ex: instructions, installation, compatibility)
- **website**: Was the website easy to navigate? Did they find the interface appealing?
- **shipping**: Shipping is outside of the control of customer support, so it deserves its own category.

The model should perform three tasks:
1. Determine which of the above topics are discussed in the review.
2. Evaluate the sentiment on a float scale from 1 to -1.
3. Evaluate the subjectivity on a scale from 0 to 1.

In [48]:
import json
from typing import List, Union
from pydantic import BaseModel, Field
from transformers import pipeline
import torch
from textblob import TextBlob

import pandas as pd
from pathlib import Path

device = 0 if torch.cuda.is_available() else -1

In [2]:
class TopicSentiment(BaseModel):
    topic: str
    sentiment_score: float = Field(ge=-1.0, le=1.0)

class ProductReviewSentiment(BaseModel):
    overall_sentiment_score: float = Field(ge=-1.0, le=1.0)
    subjectivity_score: float = Field(ge=-1.0, le=1.0)
    topics: List[TopicSentiment]

In [ ]:
topic_classifier = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli", 
    device=device
)

In [ ]:
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
    top_k=None,
    device=device
)

In [20]:
def calculate_sentiment(text: str) -> float:
    raw_results = sentiment_analyzer(text)
    results = raw_results[0] if isinstance(raw_results, list) else raw_results
    if isinstance(results, dict):
        results = [results]

    label_map = {
        'label_0': 'negative',
        'label_1': 'neutral',
        'label_2': 'positive'
    }
    scores = {
        label_map.get(result['label'].lower(), result['label'].lower()): result['score']
        for result in results
    }

    return scores['positive'] - scores['negative']

In [43]:
def _analyze_single_review(
        review_text: str,
        candidate_topics: List[str],
        hypothesis_template: str,
        topic_threshold: float = 0.4,
        min_margin_to_baseline: float = 0.1,
        print_topic_predictions: bool = False
    ) -> ProductReviewSentiment:
    overall_sentiment = calculate_sentiment(review_text)
    
    # use the TextBlob library to evaluate the subjectivity in the review
    overall_subjectivity = TextBlob(review_text).sentiment.subjectivity
    
    topic_predictions = topic_classifier(
        review_text,
        candidate_labels=candidate_topics,
        hypothesis_template=hypothesis_template,
        multi_label=True
    )
    topic_predictions = dict(zip(topic_predictions['labels'], topic_predictions['scores']))
    if print_topic_predictions:
        print(topic_predictions)

    baseline_score = topic_predictions.get(candidate_topics[-1], 0.0)

    detected_topics = []
    for label, score in topic_predictions.items():
        if label == candidate_topics[-1]:
            continue
        if score >= topic_threshold and score > baseline_score + min_margin_to_baseline:
            detected_topics.append(label)
    
    detected_topics_sentiment = []
    
    # split the review into sentences to only evaluate the sentiment of the sentence that is talking about the topic
    sentences = [s.strip() for s in review_text.replace("!", ".").replace("?", ".").split(".") if s.strip()]
    
    for label in detected_topics:
        if len(sentences) > 1:
            # find the sentence with the highest topic score
            sentence_classification = topic_classifier(sentences, candidate_labels=[label], multi_label=False)

            # Handle single vs multiple sentence output formats from pipeline
            if isinstance(sentence_classification, dict):
                sentence_classification = [sentence_classification]
            
            scored_sentences = zip(sentences, [res['scores'][0] for res in sentence_classification])
            best_sentence = max(scored_sentences, key=lambda item: item[1])[0]
        else:
            best_sentence = review_text
        
        # calculate the polarity of just the sentence with the best score
        topic_sentiment = calculate_sentiment(best_sentence)
        
        detected_topics_sentiment.append(
            TopicSentiment(topic=label, sentiment_score=topic_sentiment)
        )
            
    return ProductReviewSentiment(
        overall_sentiment_score=overall_sentiment,
        subjectivity_score=overall_subjectivity,
        topics=detected_topics_sentiment
    )


def analyze_review(
        review_input: Union[str, pd.DataFrame],
        candidate_topics: List[str],
        hypothesis_template: str,
        topic_threshold: float = 0.4,
        min_margin_to_baseline: float = 0.1,
        review_column: str = 'review_text',
        print_topic_predictions: bool = False
    ) -> Union[ProductReviewSentiment, pd.DataFrame]:
    if isinstance(review_input, str):
        return _analyze_single_review(
            review_input,
            candidate_topics=candidate_topics,
            hypothesis_template=hypothesis_template,
            topic_threshold=topic_threshold,
            min_margin_to_baseline=min_margin_to_baseline,
            print_topic_predictions=print_topic_predictions
        )

    if isinstance(review_input, pd.DataFrame):
        if review_column not in review_input.columns:
            raise ValueError(f"Dataframe must contain a '{review_column}' column.")

        predictions = [
            _analyze_single_review(
                review_text,
                candidate_topics=candidate_topics,
                hypothesis_template=hypothesis_template,
                topic_threshold=topic_threshold,
                min_margin_to_baseline=min_margin_to_baseline,
                print_topic_predictions=print_topic_predictions
            ).model_dump()
            for review_text in review_input[review_column]
        ]
        return pd.DataFrame(predictions, index=review_input.index)

    raise TypeError('review_input must be a string or pandas DataFrame.')

In [37]:
# from the writeup above
topics_list = [
    'the quality of the product',
    'customer service experience',
    'the ease of use of the product',
    'website user experience',
    'product shipping user experience',
    'unrelated and other topics'# catch-all clause
]

hypothesis_template = 'This customer review explicitly discusses {}.'

In [38]:
# retrieve a test review
data_path = Path.cwd().parent / 'data'
reviews = pd.read_csv(data_path / 'customer_reviews.csv')
reviews.head()

,review_id,review_text
0,REV-1000,Fixed my car's issue immediately. Tracking was...
1,REV-1001,The part was a perfect fit! Shipping was incre...
2,REV-1002,"The part was a perfect fit! However, the box w..."
3,REV-1003,The instructions were terrible. To make matter...
4,REV-1004,"Amazing quality, feels very solid. Shipping wa..."


In [44]:
# test the sentiment analysis pipeline with a positive example
sample_review = reviews.at[0, 'review_text']
print(sample_review)

sentiment_prediction = analyze_review(
    sample_review,
    candidate_topics=topics_list,
    hypothesis_template=hypothesis_template,
    print_topic_predictions=True
)
print(json.dumps(sentiment_prediction.model_dump(), indent=2))

Fixed my car's issue immediately. Tracking was accurate and easy.
{'customer service experience': 0.9642860889434814, 'the ease of use of the product': 0.9574390053749084, 'the quality of the product': 0.7843152284622192, 'product shipping user experience': 0.6110875010490417, 'website user experience': 0.34255799651145935, 'unrelated and other topics': 0.23956714570522308}
{
  "overall_sentiment_score": 0.689336908981204,
  "subjectivity_score": 0.5555555555555555,
  "topics": [
    {
      "topic": "customer service experience",
      "sentiment_score": 0.35224656760692596
    },
    {
      "topic": "the ease of use of the product",
      "sentiment_score": 0.7116939276456833
    },
    {
      "topic": "the quality of the product",
      "sentiment_score": 0.7116939276456833
    },
    {
      "topic": "product shipping user experience",
      "sentiment_score": 0.7116939276456833
    }
  ]
}


In [40]:
# test the pipeline with a mixed example
sample_review = reviews.at[2, 'review_text']
print(sample_review)

sentiment_prediction = analyze_review(
    sample_review,
    candidate_topics=topics_list,
    hypothesis_template=hypothesis_template,
    print_topic_predictions=True
)
print(json.dumps(sentiment_prediction.model_dump(), indent=2))

The part was a perfect fit! However, the box was completely crushed when it got here.
{'product shipping user experience': 0.6399534344673157, 'customer service experience': 0.4858860969543457, 'website user experience': 0.37784647941589355, 'unrelated and other topics': 0.23273877799510956, 'the quality of the product': 0.06712212413549423, 'the ease of use of the product': 0.017451880499720573}
{
  "overall_sentiment_score": 0.6087254211306572,
  "subjectivity_score": 0.5,
  "topics": [
    {
      "topic": "product shipping user experience",
      "sentiment_score": -0.6803690660744905
    },
    {
      "topic": "customer service experience",
      "sentiment_score": 0.9256031261757016
    }
  ]
}


In [41]:
# test the pipeline with a negative example
sample_review = reviews.at[3, 'review_text']
print(sample_review)

sentiment_prediction = analyze_review(
    sample_review,
    candidate_topics=topics_list,
    hypothesis_template=hypothesis_template,
    print_topic_predictions=True
)
print(json.dumps(sentiment_prediction.model_dump(), indent=2))

The instructions were terrible. To make matters worse, took almost three weeks to arrive.
{'customer service experience': 0.9053499102592468, 'website user experience': 0.7717331051826477, 'unrelated and other topics': 0.2592225968837738, 'product shipping user experience': 0.23845991492271423, 'the quality of the product': 0.004943552892655134, 'the ease of use of the product': 0.0005355412140488625}
{
  "overall_sentiment_score": -0.9520054506137967,
  "subjectivity_score": 0.8,
  "topics": [
    {
      "topic": "customer service experience",
      "sentiment_score": -0.9204095955938101
    },
    {
      "topic": "website user experience",
      "sentiment_score": -0.9204095955938101
    }
  ]
}


In [45]:
# batch run on the entire dataset
batch_predictions = analyze_review(
    reviews,
    candidate_topics=topics_list,
    hypothesis_template=hypothesis_template
)

reviews_with_sentiment = reviews.join(batch_predictions.add_prefix('sentiment_'))
reviews_with_sentiment.head()

,review_id,review_text,sentiment_overall_sentiment_score,sentiment_subjectivity_score,sentiment_topics
0,REV-1000,Fixed my car's issue immediately. Tracking was...,0.689337,0.555556,"[{'topic': 'customer service experience', 'sen..."
1,REV-1001,The part was a perfect fit! Shipping was incre...,0.980484,0.575000,"[{'topic': 'the quality of the product', 'sent..."
2,REV-1002,"The part was a perfect fit! However, the box w...",0.608725,0.500000,"[{'topic': 'product shipping user experience',..."
3,REV-1003,The instructions were terrible. To make matter...,-0.952005,0.800000,"[{'topic': 'customer service experience', 'sen..."
4,REV-1004,"Amazing quality, feels very solid. Shipping wa...",0.976369,0.482500,"[{'topic': 'the quality of the product', 'sent..."


In [ ]:
# export predictions alongside the original review text
prediction_export = reviews[['review_text']].join(batch_predictions)
output_path = data_path / 'sentiment_predictions.json'

with output_path.open('w', encoding='utf-8') as output_file:
    json.dump(
        prediction_export.to_dict(orient='records'),
        output_file,
        indent=2
    )

print(f'Exported {len(prediction_export)} predictions to {output_path}')